# 03 — Data cleaning and quality assessment

## Objective
Apply only cleaning supported by source evidence: trim and uppercase parameter/column labels, convert year and month values safely, map the documented `-999` sentinel to null, parse dates, normalize known city spelling variants, and detect duplicate records. No raw file is edited; no statistical outlier is dropped and no missing climate value is imputed.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name.lower()=="notebooks" else Path.cwd()
sys.path.insert(0,str(ROOT/"src"))
from climate_pipeline.config import RAW_DIR, PROJECT_DIR, FINAL_DAILY_COLUMNS
from climate_pipeline import run_pipeline, validate_city_monthly, validate_daily_schema
pd.set_option("display.max_columns",60)


## Inspect cleaning rules on source observations

The `cleaning.py` module provides column and value standardization and is used by the ingestion parser. Here we quantify its output and audit file-level schema, size, and date information before saving persistent intermediate data.

In [2]:
from climate_pipeline.ingest import discover_files, read_all_points
from climate_pipeline.cleaning import clean_source_values
from climate_pipeline.profiling import profile_frame
inventory=discover_files(RAW_DIR)
monthly_long,annual_long=read_all_points(inventory,RAW_DIR)
display(inventory[["source_file","city","measurement_point_id","number_of_rows","number_of_columns","schema_signature","date_min","date_max"]].head())
summary,field_profile=profile_frame(monthly_long,"monthly point long")
print(summary)
display(field_profile)


,source_file,city,measurement_point_id,number_of_rows,number_of_columns,schema_signature,date_min,date_max
0,ahemdabad_1.csv,Ahmedabad,1,390,15,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,2000-01-01,2025-12-31
1,ahemdabad_2.csv,Ahmedabad,2,390,15,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,2000-01-01,2025-12-31
2,ahemdabad_3.csv,Ahmedabad,3,390,15,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,2000-01-01,2025-12-31
3,ahemdabad_4.csv,Ahmedabad,4,390,15,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,2000-01-01,2025-12-31
4,ahemdabad_5.csv,Ahmedabad,5,390,15,PARAMETER|YEAR|JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG...,2000-01-01,2025-12-31


{'dataset': 'monthly point long', 'rows': 416520, 'columns': 11, 'exact_duplicate_rows': 0}


,column,dtype,missing_count,missing_pct,unique_values
0,city_key,str,0,0.00,8
1,city,str,0,0.00,8
2,measurement_point_id,int64,0,0.00,20
3,source_file,str,0,0.00,89
4,date,datetime64[us],0,0.00,312
5,PARAMETER,string,0,0.00,15
6,value,float64,3204,0.77,17433
7,latitude,float64,0,0.00,89
8,longitude,float64,0,0.00,89
9,summary_period,str,0,0.00,1


## Cleaning decisions and persisted layers

Exact duplicate and city-point-period-parameter rows are investigated before deduplication. The raw files have no identified conflicts in this collection. The pipeline saves cleaned long and wide point data plus annual summaries in `data/interim/`.

In [3]:
results=run_pipeline(RAW_DIR,ROOT)
print(results["summary"])
print("Interim tables:")
for path in sorted((ROOT/"data"/"interim").glob("*.csv")): print(path.name,path.stat().st_size,"bytes")
print("Raw inputs untouched; no missing-value imputation or outlier removal was applied.")


{'raw_files': 89, 'cities': 8, 'measurement_points': 89, 'raw_rows': 34710, 'cleaned_point_parameter_month_rows': 416520, 'annual_summary_rows': 34710, 'point_month_rows': 27768, 'city_month_rows': 2496, 'exact_duplicate_rows_detected': 0, 'duplicate_key_rows_detected': 0, 'invalid_value_records_removed': 0, 'date_start': '2000-01-01', 'date_end': '2025-12-01', 'derived_fields': ['location_id', 'representative coordinates', 'rainfall_anomaly_mm', 'temperature_anomaly_c'], 'unavailable_fields': ['daily observation grain', 'district', 'humidity_pct', 'aqi', 'pm25', 'ndvi']}
Interim tables:
point_annual_summary_long.csv 3389973 bytes
point_monthly_long.csv 41103640 bytes
point_monthly_wide.csv 4434100 bytes
Raw inputs untouched; no missing-value imputation or outlier removal was applied.


## Quality interpretation

The data are structurally consistent across the 89 source files. The source period is 2000–2025 at monthly and annual summary grain. Cleaning and quality summaries are written under `reports/`, including a data dictionary and explicit missingness report.